# I₂ — TDDFT, TDDFT+SOC, QED-TDDFT, QED-TDDFT+SOC

Iodine is a heavy closed-shell diatomic: **spin–orbit coupling is large**, so borrowed oscillator strength and polariton mixing with triplets are the point of this demo.

Pipeline (Level A — ordinary RKS ground state; SOC / cavity are post-SCF):

1. **TDDFT (TDA)** — closed-shell singlets and triplets  
2. **TDDFT + SI-SOC** — one-electron state-interaction mixing (`casidapy.utils.soc`)  
3. **QED-TDDFT** — Pauli–Fierz QED-TDA on the **singlet** manifold only (`solve_qed_tda`)  
4. **QED-TDDFT + SOC** — truncated Pauli–Fierz on the **SI-SOC** eigenbasis ⊗ {0,1} photons (`solve_soc_qed_pf`)

Iodine needs an **ECP basis** (all-electron `sto-3g` is too small: `Nocc > Nmo`). Default: `def2-svp` + matching ECP + `pbe`.


In [1]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, dft

from casidapy import (
    extract_gto_kernel,
    run_casida,
    solve_soc_si,
    solve_qed_tda,
    solve_soc_qed_pf,
    QEDOptions,
)

HA_TO_EV = 27.211386245988

# --- defaults (edit freely) ---
# I₂: must use an ECP basis. All-electron sto-3g has too few AOs (Nocc=53 > Nmo).
XC = "pbe"
BASIS = "def2-svp"
ECP = "def2-svp"          # required for I; omit only with a large all-electron basis
R_II = 2.666              # Å, near experimental r_e
N_S = 4
N_T = 4
LAM = 0.05
POL = (0.0, 0.0, 1.0)     # along the I–I axis (z)

I2_ATOM = f"""
I  0.000000  0.000000  0.000000
I  0.000000  0.000000  {R_II:.6f}
"""

print(f"defaults: {XC}/{BASIS} (ecp={ECP}), r(I–I)={R_II} Å, λ={LAM}, pol=z")


--------------------------------------------------------------------------
No OpenFabrics connection schemes reported that they were able to be
used on a specific port.  As such, the openib BTL (OpenFabrics
support) will be disabled for this port.

  Local host:           amareln
  Local device:         mlx5_0
  Local port:           1
  CPCs attempted:       rdmacm, udcm
--------------------------------------------------------------------------


defaults: pbe/def2-svp (ecp=def2-svp), r(I–I)=2.666 Å, λ=0.05, pol=z


## 1. SCF + bare TDDFT (singlet & triplet TDA)


In [2]:
mol = gto.M(atom=I2_ATOM, basis=BASIS, ecp=ECP, spin=0, verbose=0)
print(
    f"nao={mol.nao_nr()}  nelectron={mol.nelectron}  "
    f"(need nocc={mol.nelectron//2} ≤ nao)"
)

mf = dft.RKS(mol)
mf.xc = XC
mf.grids.level = 3
mf.level_shift = 0.2
e_scf = mf.kernel()
if not mf.converged:
    mf.level_shift = 0.0
    e_scf = mf.kernel()
print(f"SCF E = {e_scf:.8f} Ha   converged={mf.converged}")

ks, opts_s = extract_gto_kernel(
    mf, n_states=N_S, tda=True, use_df=False, spin_state="singlet",
)
kt, opts_t = extract_gto_kernel(
    mf, n_states=N_T, tda=True, use_df=False, spin_state="triplet",
)
opts_s.solver_method = opts_t.solver_method = "eigsh"
res_s = run_casida(ks, opts_s)
res_t = run_casida(kt, opts_t)

print("\nSinglet TDA:")
for i, w in enumerate(res_s.omega):
    print(f"  S{i+1}: {w*HA_TO_EV:8.3f} eV   f={res_s.f[i]:.4e}")
print("Triplet TDA:")
for i, w in enumerate(res_t.omega):
    print(f"  T{i+1}: {w*HA_TO_EV:8.3f} eV   f={res_t.f[i]:.4e}")

if np.any(res_t.omega < 0):
    print("\nwarning: negative triplet roots — RKS may be unstable at this geometry/XC.")


nao=52  nelectron=50  (need nocc=25 ≤ nao)


[amareln.amareln.rutgers.edu:238706] 1 more process has sent help message help-mpi-btl-openib-cpc-base.txt / no cpcs for port
[amareln.amareln.rutgers.edu:238706] Set MCA parameter "orte_base_help_aggregate" to 0 to see all help / error messages


SCF E = -595.38759610 Ha   converged=True


KeyboardInterrupt: 

## 2. TDDFT + SI-SOC

One-electron state-interaction SOC on the singlet ⊕ Cartesian-triplet manifold.  
For I₂ the mixing (and borrowed `f`) should be much stronger than for H₂CO.


In [ ]:
soc = solve_soc_si(res_s, res_t, ks, include_ground=False)

print("SI-SOC mixed roots:")
print(f"{'i':>4} {'ω (eV)':>10} {'S wt':>8} {'T wt':>8} {'f':>12}")
print("-" * 48)
for i, w in enumerate(soc.omega):
    print(
        f"{i:4d} {w*HA_TO_EV:10.3f} {soc.singlet_weight[i]:8.3f} "
        f"{soc.triplet_weight[i]:8.3f} {soc.f[i]:10.4e}"
    )


## 3. QED-TDDFT (singlets only) and QED-TDDFT+SOC

- **QED-TDDFT:** full closed-shell Pauli–Fierz TDA on the singlet Casida space (`solve_qed_tda`), including DSE by default.  
- **QED-TDDFT+SOC:** truncated PF on SI-SOC roots ⊕ `|S₀⟩`, electronic ⊗ {0,1} photons (`solve_soc_qed_pf`).

Cavity frequency defaults to the **brightest singlet** (largest `f`). Override with a fixed `OMEGA_C_EV` if you want a specific resonance.


In [ ]:
OMEGA_C_EV = None  # e.g. 2.5 to fix ω_c; None → brightest singlet

if OMEGA_C_EV is None:
    k_bright = int(np.argmax(np.asarray(res_s.f, dtype=float)))
    omega_c = float(res_s.omega[k_bright])
    print(
        f"ω_c = {omega_c*HA_TO_EV:.3f} eV  (brightest S{k_bright+1}, "
        f"f={res_s.f[k_bright]:.3e})"
    )
else:
    omega_c = float(OMEGA_C_EV) / HA_TO_EV
    print(f"ω_c = {omega_c*HA_TO_EV:.3f} eV  (user-fixed)")

lam_vec = np.asarray(POL, float) * LAM

qed_opts = QEDOptions(
    lam_scalar=LAM,
    polarization=POL,
    omega_c=omega_c,
    nstates=min(8, ks.n_trans + 1),
    include_dse=True,
    coherent_state=True,
)
qed_s = solve_qed_tda(ks, options=qed_opts)

qed_soc = solve_soc_qed_pf(
    soc,
    lam_vec=lam_vec,
    omega_c=omega_c,
    nstates=None,
    include_ground_slot=True,
    include_dse=True,
    prefer_bright=False,
)

print("\nQED-TDDFT (singlets only):")
for i, w in enumerate(qed_s.omega):
    print(f"  {i}: {w*HA_TO_EV:8.3f} eV   |m|²={qed_s.photon_frac[i]:.3f}")

print("\nQED-TDDFT+SOC (PF on SI-SOC ⊗ {0,1}):")
for i, w in enumerate(qed_soc["omega"][:16]):
    print(f"  {i}: {w*HA_TO_EV:8.3f} eV   |m|²={qed_soc['photon_frac'][i]:.3f}")


## 4. Stick spectra

Compare bare electronic sticks, SOC-mixed sticks, singlet QED-TDA, and SOC+PF.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), sharey=False)

# --- left: electronic / SOC ---
ax = axes[0]
ymax = 1.05
ax.vlines(res_s.omega * HA_TO_EV, 0, np.clip(res_s.f / max(res_s.f.max(), 1e-30), 0, 1),
          colors="C0", lw=2.0, label="TDA singlets (f)")
ax.vlines(res_t.omega * HA_TO_EV, 0, 0.25 * np.ones_like(res_t.omega),
          colors="C3", lw=1.4, linestyles=":", label="TDA triplets")
f_soc = np.asarray(soc.f, dtype=float)
ax.vlines(soc.omega * HA_TO_EV, 0, np.clip(f_soc / max(f_soc.max(), 1e-30), 0, 1),
          colors="C2", lw=1.6, alpha=0.85, label="SI-SOC (f)")
ax.set_xlabel("excitation energy (eV)")
ax.set_ylabel("relative intensity")
ax.set_title("TDDFT vs TDDFT+SOC")
ax.set_ylim(0, ymax)
ax.legend(fontsize=8, loc="best")
ax.grid(True, alpha=0.3)

# --- right: QED (use oscillator strengths, not photon weight) ---
ax = axes[1]
f_qed = np.asarray(qed_s.f, dtype=float)
ax.vlines(qed_s.omega * HA_TO_EV, 0, np.clip(f_qed / max(f_qed.max(), 1e-30), 0, 1),
          colors="C0", lw=1.8, label="QED-TDDFT (S only)")
w_pf = qed_soc["omega"]
f_pf = np.asarray(qed_soc["f"], dtype=float)
mask = w_pf * HA_TO_EV > 0.05
ax.vlines(w_pf[mask] * HA_TO_EV, 0, np.clip(f_pf[mask] / max(f_pf.max(), 1e-30), 0, 1),
          colors="C3", lw=1.5, alpha=0.9, label="QED-TDDFT+SOC (f)")
ax.axvline(omega_c * HA_TO_EV, color="k", ls=":", lw=1.2, label=r"$\omega_c$")
ax.set_xlabel("excitation energy (eV)")
ax.set_ylabel("relative oscillator strength")
ax.set_title(f"QED (λ={LAM})")
ax.legend(fontsize=8, loc="best")
ax.grid(True, alpha=0.3)

fig.suptitle(f"I₂ — {XC}/{BASIS}, r={R_II} Å", y=1.02)
fig.tight_layout()
plt.show()


## Notes

- **Basis:** Iodine requires an ECP (`def2-svp` + `ecp='def2-svp'`). All-electron `sto-3g` fails with `Nocc > Nmo`.
- **SOC:** 1e SI only (`int1e_ia01p`); 2e SOC omitted — qualitative, can overestimate mixing for heavy atoms.
- **QED-TDDFT** is the dense singles⊕photon TDA matrix (DSE on). **QED-TDDFT+SOC** is truncated PF on mixed electronic eigenstates.
- If triplets go negative, try a slightly shorter `R_II`, hybrid XC (`pbe0`), or more SCF damping.
